# Nethravathi Flood Threshold Mapping + GNN Assessment


**Study area:** Upper Nethravathi Basin, Karnataka, India

**Flood event simulated:** SW Monsoon 2018 (Aug 8–18)

**Calibration year:** 2014 (CHIRPS rainfall + CWC discharge — year-matched)


In [ ]:
# ================================================================
# CELL 1: INSTALL PACKAGES
# Run once per Colab session. Installs all required libraries.
# ================================================================

!pip install rasterio geopandas pysheds scipy scikit-image --quiet

import os, json, glob, re, warnings, gzip, shutil, requests
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.warp import Resampling as WarpResampling
from rasterio.mask import mask as rio_mask
from scipy import ndimage
from scipy.ndimage import distance_transform_edt, gaussian_filter
from shapely.geometry import box, mapping
from datetime import date, timedelta
import pandas as pd
warnings.filterwarnings('ignore')

print('All packages loaded successfully')

In [ ]:
# ================================================================
# CELL 2: PROJECT SETUP — PATHS AND FOLDERS
# Run every time after reconnection. Creates folder structure
# and defines all file paths used throughout the notebook.
# ================================================================

PROJECT_ROOT = '/content/Nethravathi_Flood'

for folder in [
    'data/raw/dem', 'data/raw/rainfall', 'data/raw/rainfall_2014',
    'data/raw/lulc', 'data/raw/soil', 'data/raw/discharge',
    'data/processed', 'outputs/flood_maps',
    'outputs/plots', 'outputs/rasters', 'models', 'report'
]:
    os.makedirs(f'{PROJECT_ROOT}/{folder}', exist_ok=True)

# Raw input paths
DEM_PATH       = f'{PROJECT_ROOT}/data/raw/dem/srtm_nethravathi_30m.tif'
LULC_ESA_RAW   = f'{PROJECT_ROOT}/data/raw/lulc/ESA_WorldCover_10m_2021_v200_N12E075_Map.tif'
SOIL_RAW       = f'{PROJECT_ROOT}/data/raw/soil/Netra_soil.tif'
RAIN_DIR       = f'{PROJECT_ROOT}/data/raw/rainfall'
RAIN_DIR_2014  = f'{PROJECT_ROOT}/data/raw/rainfall_2014'
DISCHARGE_PATH = f'{PROJECT_ROOT}/data/raw/discharge/discharge.txt'

# Processed output paths
PROC_DIR       = f'{PROJECT_ROOT}/data/processed'
OUTPUTS_DIR    = f'{PROJECT_ROOT}/outputs'
DEM_UTM_PATH   = f'{PROC_DIR}/srtm_utm43n.tif'
LULC_UTM       = f'{PROC_DIR}/lulc_esa_utm43n.tif'
SOIL_UTM       = f'{PROC_DIR}/soil_matched.tif'
CN_PATH        = f'{PROC_DIR}/curve_number.tif'

# AOI: Upper Nethravathi sub-basin
# NOTE: This covers ~800 km² of the upper basin.
# The full Nethravathi at Bantwal drains ~3657 km².
# This area mismatch is a documented structural limitation.
MINX, MINY, MAXX, MAXY = 75.55, 12.80, 76.05, 13.20
AOI_AREA_KM2   = 800.0    # upper sub-basin (modelled domain)
FULL_BASIN_KM2 = 3657.0   # full Nethravathi at Bantwal (literature)
DST_CRS        = 'EPSG:32643'

bbox = {'minx':MINX,'miny':MINY,'maxx':MAXX,'maxy':MAXY}
with open(f'{PROC_DIR}/bbox.json','w') as f:
    json.dump(bbox, f)

print('Project setup complete')
print(f'AOI      : {MINX}E-{MAXX}E, {MINY}N-{MAXY}N')
print(f'AOI area : ~{AOI_AREA_KM2:.0f} km2 (upper sub-basin)')
print(f'Note     : Full Bantwal catchment = {FULL_BASIN_KM2:.0f} km2')

In [ ]:
# ================================================================
# CELL 3: UPLOAD INPUT FILES
# Upload these files from your computer:
#   - srtm_nethravathi_30m.tif
#   - Netra_soil.tif
#   - discharge.txt  (CWC Bantwal 1981-2015)
#
# CHIRPS rainfall is downloaded automatically in Cell 4.
# ESA WorldCover is downloaded automatically in Cell 5.
# ================================================================

from google.colab import files

# --- DEM ---
print('STEP 1: Upload srtm_nethravathi_30m.tif')
uploaded = files.upload()
for fname, content in uploaded.items():
    with open(DEM_PATH,'wb') as f: f.write(content)
    print(f'  Saved: {DEM_PATH}')

# --- Soil ---
print('\nSTEP 2: Upload Netra_soil.tif')
uploaded = files.upload()
for fname, content in uploaded.items():
    with open(SOIL_RAW,'wb') as f: f.write(content)
    print(f'  Saved: {SOIL_RAW}')

# --- Discharge ---
# Format: row_number TAB FLOW_OUT_DOY_YEAR TAB discharge_m3s
# This file is used in Cell 13 for calibration.
print('\nSTEP 3: Upload discharge.txt (CWC Bantwal 1981-2015)')
uploaded = files.upload()
for fname, content in uploaded.items():
    with open(DISCHARGE_PATH,'wb') as f: f.write(content)
    print(f'  Saved: {DISCHARGE_PATH}')

print('\nAll files uploaded. CHIRPS and ESA downloaded automatically in next cells.')

In [ ]:
# ================================================================
# CELL 4: DOWNLOAD CHIRPS RAINFALL
# Downloads two CHIRPS monsoon seasons:
#   A) 2018 Jun-Sep: for flood EVENT simulation (Aug 8-18)
#   B) 2014 Jun-Sep: for CALIBRATION (within discharge record)
# Each file ~200KB compressed. Total ~50MB. Takes ~10 minutes.
# ================================================================

BASE_URL = (
    'https://data.chc.ucsb.edu/products/CHIRPS-2.0/'
    'global_daily/tifs/p05/{year}/'
    'chirps-v2.0.{year}.{month:02d}.{day:02d}.tif.gz'
)

def download_chirps_season(year, rain_dir):
    os.makedirs(rain_dir, exist_ok=True)
    dates = []
    d = date(year, 6, 1)
    while d <= date(year, 9, 30):
        dates.append(d)
        d += timedelta(days=1)

    existing = len(glob.glob(f'{rain_dir}/*.tif.gz'))
    if existing >= 122:
        print(f'  {year}: already downloaded ({existing} files)')
        return

    print(f'  Downloading {year} monsoon ({len(dates)} files)...')
    success, failed = 0, []
    for d in dates:
        url   = BASE_URL.format(year=d.year, month=d.month, day=d.day)
        fname = f'chirps-v2.0.{d.year}.{d.month:02d}.{d.day:02d}.tif.gz'
        fpath = f'{rain_dir}/{fname}'
        if os.path.exists(fpath):
            success += 1
            continue
        try:
            r = requests.get(url, timeout=30, stream=True)
            if r.status_code == 200:
                with open(fpath,'wb') as f:
                    for chunk in r.iter_content(8192): f.write(chunk)
                success += 1
                if success % 30 == 0:
                    print(f'    {success}/{len(dates)} done...')
            else:
                failed.append(str(d))
        except:
            failed.append(str(d))
    print(f'  {year}: {success} ok, {len(failed)} failed')

print('Downloading CHIRPS rainfall...')
download_chirps_season(2018, RAIN_DIR)       # flood event
download_chirps_season(2014, RAIN_DIR_2014)  # calibration year
print('Download complete')

In [ ]:
# ================================================================
# CELL 5: DOWNLOAD ESA WORLDCOVER LULC
# Downloads ESA WorldCover 10m land use tile for 2021.
# Tile N12E075 covers our full AOI. ~128 MB.
# ================================================================

url = ('https://esa-worldcover.s3.amazonaws.com/v200/2021/map/'
       'ESA_WorldCover_10m_2021_v200_N12E075_Map.tif')

if os.path.exists(LULC_ESA_RAW):
    print('ESA WorldCover already downloaded')
else:
    print('Downloading ESA WorldCover (~128 MB)...')
    r = requests.get(url, stream=True, timeout=300)
    if r.status_code == 200:
        with open(LULC_ESA_RAW,'wb') as f:
            for chunk in r.iter_content(65536): f.write(chunk)
        print(f'Downloaded: {os.path.getsize(LULC_ESA_RAW)/1e6:.1f} MB')
    else:
        print(f'Failed: {r.status_code} — upload tile manually')

with rasterio.open(LULC_ESA_RAW) as src:
    d = src.read(1)
    print(f'LULC classes: {sorted(np.unique(d[d!=0]).tolist())}')

In [ ]:
# ================================================================
# CELL 6: VERIFY AND REPROJECT DEM
# Converts DEM from geographic coordinates (EPSG:4326)
# to UTM Zone 43N (EPSG:32643) in metres.
# Required for all area and distance calculations.
# ================================================================

with rasterio.open(DEM_PATH) as src:
    dem_raw = src.read(1).astype(np.float32)
    if src.nodata: dem_raw[dem_raw==src.nodata] = np.nan
    valid = dem_raw[~np.isnan(dem_raw)]
    print(f'Input DEM: {src.width}x{src.height}px, '
          f'{valid.min():.0f}-{valid.max():.0f}m, CRS={src.crs}')

with rasterio.open(DEM_PATH) as src:
    transform_utm, width_utm, height_utm = calculate_default_transform(
        src.crs, DST_CRS, src.width, src.height, *src.bounds)
    meta_utm = src.meta.copy()
    meta_utm.update({'crs':DST_CRS,'transform':transform_utm,
                     'width':width_utm,'height':height_utm,'nodata':-9999})
    with rasterio.open(DEM_UTM_PATH,'w',**meta_utm) as dst:
        reproject(source=rasterio.band(src,1),
                  destination=rasterio.band(dst,1),
                  src_transform=src.transform, src_crs=src.crs,
                  dst_transform=transform_utm, dst_crs=DST_CRS,
                  resampling=WarpResampling.bilinear)

with rasterio.open(DEM_UTM_PATH) as src:
    dem_utm  = src.read(1).astype(np.float32)
    meta_utm = src.meta.copy()
dem_utm[dem_utm==-9999] = np.nan
target_shape = dem_utm.shape
dem_valid    = ~np.isnan(dem_utm)

print(f'UTM DEM  : {src.width}x{src.height}px, '
      f'res={src.res[0]:.1f}m, '
      f'{np.nanmin(dem_utm):.0f}-{np.nanmax(dem_utm):.0f}m')

In [ ]:
# ================================================================
# CELL 7: PROCESS ESA WORLDCOVER LULC
# Clips global tile to AOI, reprojects to UTM 43N at 30m.
# Must match DEM grid for CN calculation.
# ================================================================

ESA_CLIPPED = f'{PROJECT_ROOT}/data/raw/lulc/esa_clipped.tif'
aoi_geom    = mapping(box(MINX, MINY, MAXX, MAXY))

with rasterio.open(LULC_ESA_RAW) as src:
    clipped, clip_t = rio_mask(src, [aoi_geom], crop=True, nodata=0)
    clip_meta = src.meta.copy()
    clip_meta.update({'height':clipped.shape[1],'width':clipped.shape[2],
                      'transform':clip_t,'nodata':0})
    with rasterio.open(ESA_CLIPPED,'w',**clip_meta) as dst:
        dst.write(clipped)

with rasterio.open(ESA_CLIPPED) as src:
    t_utm,w_utm,h_utm = calculate_default_transform(
        src.crs, DST_CRS, src.width, src.height, *src.bounds, resolution=30)
    lulc_meta = src.meta.copy()
    lulc_meta.update({'crs':DST_CRS,'transform':t_utm,
                      'width':w_utm,'height':h_utm,'nodata':0,'dtype':'uint8'})
    with rasterio.open(LULC_UTM,'w',**lulc_meta) as dst:
        reproject(source=rasterio.band(src,1),
                  destination=rasterio.band(dst,1),
                  src_transform=src.transform, src_crs=src.crs,
                  dst_transform=t_utm, dst_crs=DST_CRS,
                  resampling=WarpResampling.nearest)

with rasterio.open(LULC_UTM) as src:
    lc = src.read(1)
print(f'LULC: classes={sorted(np.unique(lc[lc!=0]).tolist())}, '
      f'coverage={(lc!=0).sum()/lc.size*100:.1f}%')

In [ ]:
# ================================================================
# CELL 8: PROCESS SOIL
# Reprojects soil map to exactly match LULC grid.
# Both must have identical dimensions for CN grid calculation.
# ================================================================

with rasterio.open(LULC_UTM) as lulc_src:
    lulc_shape     = (lulc_src.height, lulc_src.width)
    lulc_transform = lulc_src.transform
    lulc_crs       = lulc_src.crs

with rasterio.open(SOIL_RAW) as soil_src:
    soil_matched = np.zeros(lulc_shape, dtype=np.float32)
    reproject(source=rasterio.band(soil_src,1),
              destination=soil_matched,
              src_transform=soil_src.transform, src_crs=soil_src.crs,
              dst_transform=lulc_transform, dst_crs=lulc_crs,
              resampling=WarpResampling.nearest)
    out_meta = soil_src.meta.copy()
    out_meta.update({'height':lulc_shape[0],'width':lulc_shape[1],
                     'transform':lulc_transform,'crs':lulc_crs,'dtype':'float32'})
    with rasterio.open(SOIL_UTM,'w',**out_meta) as dst:
        dst.write(soil_matched,1)

with rasterio.open(SOIL_UTM) as src:
    sd = src.read(1)
print(f'Soil matched to LULC grid: {lulc_shape}')
print(f'Soil codes: {sorted(np.unique(sd[sd!=0]).astype(int).tolist())}')

In [ ]:
# ================================================================
# CELL 9: BUILD SCS CURVE NUMBER GRID
# CN controls how much rainfall becomes runoff.
# Built from ESA LULC × Hydrologic Soil Group lookup (USDA TR-55).
# Soil codes map to HSG:
#   3656 → HSG B (loam)
#   3817 → HSG C (laterite/clay)  <- dominant in Western Ghats
#   3824 → HSG C
#   3826 → HSG D (heavy clay)
# ================================================================

ESA_CN_TABLE = {
    10:{1:30,2:55,3:70,4:77},   # Forest
    20:{1:35,2:60,3:73,4:79},   # Shrubland
    30:{1:49,2:69,3:79,4:84},   # Grassland
    40:{1:67,2:78,3:85,4:89},   # Cropland
    50:{1:77,2:85,3:90,4:92},   # Built-up
    60:{1:77,2:86,3:91,4:94},   # Bare ground
    70:{1:100,2:100,3:100,4:100}, # Water
    80:{1:100,2:100,3:100,4:100}, # Permanent water
    90:{1:78,2:84,3:88,4:90},   # Wetland
    95:{1:78,2:84,3:88,4:90},   # Mangroves
    100:{1:77,2:86,3:91,4:94},  # Moss/lichen
}
SOIL_HSG = {0:0, 3656:2, 3817:3, 3824:3, 3826:4}
DEFAULT_CN = 72

with rasterio.open(LULC_UTM) as src: lulc = src.read(1).astype(np.float32)
with rasterio.open(SOIL_UTM) as src: soil = src.read(1).astype(np.float32)

hsg = np.full_like(soil, 3, dtype=np.int16)
for code, group in SOIL_HSG.items():
    hsg[soil==code] = group

cn_grid = np.full_like(lulc, DEFAULT_CN, dtype=np.float32)
for lc, hsg_dict in ESA_CN_TABLE.items():
    for hg, cv in hsg_dict.items():
        mask = (lulc==lc) & (hsg==hg)
        if mask.any(): cn_grid[mask] = cv

unknown = (hsg==0) & (lulc!=0)
for lc, hsg_dict in ESA_CN_TABLE.items():
    mask = unknown & (lulc==lc)
    if mask.any(): cn_grid[mask] = hsg_dict.get(3, DEFAULT_CN)

cn_grid[lulc==0] = -9999
with rasterio.open(LULC_UTM) as src: cn_meta = src.meta.copy()
cn_meta.update({'dtype':'float32','nodata':-9999,'count':1})
with rasterio.open(CN_PATH,'w',**cn_meta) as dst: dst.write(cn_grid,1)

valid_cn = cn_grid[cn_grid!=-9999]
print(f'CN grid: mean={valid_cn.mean():.1f}, range={valid_cn.min():.0f}-{valid_cn.max():.0f}')
print(f'Coverage: {len(valid_cn)/cn_grid.size*100:.1f}%')
for g, name in zip([1,2,3,4],['A','B','C','D']):
    print(f'  HSG {name}: {(hsg==g).sum()/(lulc!=0).sum()*100:.1f}%')

In [ ]:
# ================================================================
# CELL 10: TERRAIN FEATURE EXTRACTION (D8 FLOW ROUTING)
# Computes all terrain features needed by the GNN:
#   slope_deg, flow_accumulation, river_mask,
#   dist_to_river, hand (Height Above Nearest Drainage), twi
# Uses pysheds for proper pit-filled D8 routing.
# Takes ~5 minutes.
# ================================================================

!pip install pysheds --quiet
from pysheds.grid import Grid

res_m = 30.0

def save_raster(arr, filename, meta, nd=-9999.0):
    m = meta.copy()
    m.update({'dtype':'float32','nodata':nd,'count':1})
    a = np.where(np.isnan(arr), nd, arr).astype('float32')
    path = f'{PROC_DIR}/{filename}'
    with rasterio.open(path,'w',**m) as dst: dst.write(a,1)
    v = a[a!=nd]
    print(f'  Saved {filename}: {v.min():.2f} - {v.max():.2f}')
    return path

print('[1/5] Slope and aspect...')
dem_f  = np.where(np.isnan(dem_utm), np.nanmean(dem_utm), dem_utm)
dz_dy, dz_dx = np.gradient(dem_f, res_m, res_m)
slope_deg = np.degrees(np.arctan(np.sqrt(dz_dx**2+dz_dy**2)))
slope_deg[np.isnan(dem_utm)] = np.nan
save_raster(slope_deg, 'slope_deg.tif', meta_utm)

print('[2/5] D8 flow routing (pysheds)...')
grid     = Grid.from_raster(DEM_UTM_PATH)
dem_grid = grid.read_raster(DEM_UTM_PATH)
dirmap   = (64,128,1,2,4,8,16,32)
flow_dir = grid.flowdir(grid.resolve_flats(grid.fill_depressions(grid.fill_pits(dem_grid))), dirmap=dirmap)
flow_acc = grid.accumulation(flow_dir, dirmap=dirmap)
facc_np  = np.array(flow_acc).astype(np.float32)
facc_np[facc_np<=0] = np.nan
print(f'  Max accumulation: {np.nanmax(facc_np):.0f} cells')
save_raster(np.log1p(facc_np), 'flow_accumulation.tif', meta_utm)

print('[3/5] River network...')
MIN_PIX  = 2222
river    = ((facc_np>=MIN_PIX) & dem_valid).astype(np.uint8)
print(f'  River pixels: {river.sum():,} ({river.sum()/dem_valid.sum()*100:.2f}%)')
save_raster(river.astype(np.float32), 'river_mask.tif', meta_utm)
dist_r   = ndimage.distance_transform_edt(1-river) * res_m
dist_r[np.isnan(dem_utm)] = np.nan
save_raster(dist_r, 'dist_to_river.tif', meta_utm)

print('[4/5] HAND...')
_, idx   = distance_transform_edt(1-river, return_indices=True)
hand     = np.clip(dem_f - dem_f[idx[0],idx[1]], 0, None)
hand[np.isnan(dem_utm)] = np.nan
save_raster(hand, 'hand.tif', meta_utm)

print('[5/5] TWI...')
slope_r  = np.where(np.degrees(np.arctan(np.sqrt(dz_dx**2+dz_dy**2)))<0.001, 0.001, np.arctan(np.sqrt(dz_dx**2+dz_dy**2)))
twi      = np.clip(np.log((facc_np+1)/np.tan(slope_r)), 0, 25)
twi[np.isnan(dem_utm)] = np.nan
save_raster(twi, 'twi.tif', meta_utm)

print('\nHAND flood zones:')
for t, label in [(2,'Immediate'),(5,'Active'),(10,'100-yr'),(50,'Upland')]:
    print(f'  HAND<{t:>3}m ({label}): {(hand<t).sum()/dem_valid.sum()*100:.1f}%')

In [ ]:
# ================================================================
# CELL 11: PROCESS CHIRPS RAINFALL STACKS
# Processes both 2018 and 2014 CHIRPS into numpy arrays.
# 2018 stack → flood event simulation
# 2014 stack → calibration
# Output shape: (122 days, height, width)
# ================================================================

def process_chirps_stack(rain_dir, year, desc):
    gz_files  = sorted(glob.glob(f'{rain_dir}/*.tif.gz'))
    print(f'Processing {len(gz_files)} files for {year} ({desc})...')
    aoi_geom  = mapping(box(MINX, MINY, MAXX, MAXY))
    stack     = np.zeros((len(gz_files), *target_shape), dtype=np.float32)

    for i, gz in enumerate(gz_files):
        tif = gz.replace('.tif.gz','.tif')
        try:
            if not os.path.exists(tif):
                with gzip.open(gz,'rb') as fi:
                    with open(tif,'wb') as fo: shutil.copyfileobj(fi,fo)
            with rasterio.open(tif) as src:
                clipped, ct = rio_mask(src,[aoi_geom],crop=True,nodata=-9999)
                cm = src.meta.copy()
                cm.update({'height':clipped.shape[1],'width':clipped.shape[2],
                           'transform':ct,'nodata':-9999})
            tmp = tif.replace('.tif','_tmp.tif')
            with rasterio.open(tmp,'w',**cm) as t: t.write(clipped)
            with rasterio.open(tmp) as src:
                data = np.zeros(target_shape, dtype=np.float32)
                reproject(source=rasterio.band(src,1),destination=data,
                          src_transform=src.transform,src_crs=src.crs,
                          dst_transform=meta_utm['transform'],dst_crs=DST_CRS,
                          resampling=WarpResampling.bilinear)
            data[data<0] = 0
            stack[i] = data
            os.remove(tmp)
        except:
            stack[i] = 0
        if (i+1) % 30 == 0:
            print(f'  Day {i+1}/122...')

    total = stack.sum(axis=0)[dem_valid].mean()
    print(f'  {year} seasonal total: {total:.0f}mm')
    return stack

# Process 2018
stack_2018     = process_chirps_stack(RAIN_DIR, 2018, 'flood event')
total_rain     = stack_2018.sum(axis=0)
peak_event_rain = stack_2018[68:79].sum(axis=0)  # Aug 8-18
np.save(f'{PROC_DIR}/rainfall_stack.npy',  stack_2018)
np.save(f'{PROC_DIR}/total_rainfall.npy',  total_rain)
np.save(f'{PROC_DIR}/peak_event_rain.npy', peak_event_rain)

# Process 2014
stack_2014 = process_chirps_stack(RAIN_DIR_2014, 2014, 'calibration')
np.save(f'{PROC_DIR}/rainfall_stack_2014.npy', stack_2014)

print(f'\nAug 8-18 2018 total: {peak_event_rain[dem_valid].mean():.0f}mm')
print('Both rainfall stacks saved')

In [ ]:
# ================================================================
# CELL 12: RELOAD ALL VARIABLES INTO MEMORY
# Run this FIRST after any Colab disconnection.
# Reloads all saved files and redefines model functions.
# ================================================================

def load_raster(path):
    with rasterio.open(path) as src:
        d = src.read(1).astype(np.float32)
        if src.nodata is not None: d[d==src.nodata] = np.nan
        return d, src.meta.copy()

def match_shape(arr, target):
    if arr.shape == target: return arr
    from scipy.ndimage import zoom
    return zoom(arr,(target[0]/arr.shape[0],target[1]/arr.shape[1]),order=1)

def scscn_runoff(P, CN, Ia_ratio=0.2):
    """SCS-CN rainfall-runoff. Returns runoff depth Q (mm).
    Ia_ratio=0.05 used (not default 0.20) for Western Ghats
    saturated monsoon conditions."""
    CN = np.clip(CN, 1, 99)
    S  = (25400.0/CN) - 254.0
    Ia = Ia_ratio * S
    d  = P - Ia + S
    Q  = np.where((P>Ia)&(d>0), (P-Ia)**2/d, 0.0)
    return np.clip(Q, 0, None)

def estimate_flood_depth(Q_runoff, hand, slope, facc, river):
    """Physics-based flood depth surrogate using HAND attenuation."""
    flood = ((Q_runoff/1000.0)
             * np.exp(-hand/10.0)
             * np.exp(-slope/25.0)
             * (1.0 + 2.0*np.clip(facc/(facc.max()+1e-6),0,1)))
    flood = gaussian_filter(np.where(np.isnan(flood),0,flood), sigma=2)
    flood = np.where(hand<50, flood, 0)
    return np.where(dem_valid, flood, np.nan)

def save_flood_raster(arr, filename, meta):
    m = meta.copy()
    m.update({'dtype':'float32','nodata':-9999,'count':1})
    a = np.where(np.isnan(arr),-9999,arr).astype('float32')
    path = f'{OUTPUTS_DIR}/flood_maps/{filename}'
    with rasterio.open(path,'w',**m) as dst: dst.write(a,1)
    return path

# Load all rasters
dem_utm,   meta_utm = load_raster(DEM_UTM_PATH)
dem_valid           = ~np.isnan(dem_utm)
target_shape        = dem_utm.shape
cn,    _            = load_raster(CN_PATH)
hand,  _            = load_raster(f'{PROC_DIR}/hand.tif')
slope, _            = load_raster(f'{PROC_DIR}/slope_deg.tif')
facc,  _            = load_raster(f'{PROC_DIR}/flow_accumulation.tif')
river, _            = load_raster(f'{PROC_DIR}/river_mask.tif')
rain_event          = np.load(f'{PROC_DIR}/peak_event_rain.npy')
rainfall_stack      = np.load(f'{PROC_DIR}/rainfall_stack.npy')
stack_2014          = np.load(f'{PROC_DIR}/rainfall_stack_2014.npy')

for name in ['cn','hand','slope','facc','river','rain_event']:
    exec(f'{name} = match_shape({name}, target_shape)')

cn         = np.where(np.isnan(cn),    74.5, cn)
hand       = np.where(np.isnan(hand),  500,  hand)
slope      = np.where(np.isnan(slope), 5.0,  slope)
facc       = np.where(np.isnan(facc),  0.0,  facc)
rain_event = np.where(rain_event<0,    0.0,  rain_event)

os.makedirs(f'{OUTPUTS_DIR}/flood_maps', exist_ok=True)
os.makedirs(f'{OUTPUTS_DIR}/plots',      exist_ok=True)

print('All variables loaded')
print(f'  DEM        : {dem_utm.shape}')
print(f'  CN mean    : {cn[dem_valid].mean():.1f}')
print(f'  HAND max   : {hand[dem_valid].max():.0f}m')
print(f'  Rain event : {rain_event[dem_valid].mean():.1f}mm mean')
print(f'  Stack 2018 : {rainfall_stack.shape}')
print(f'  Stack 2014 : {stack_2014.shape}')

In [ ]:
# ================================================================
# CELL 13: CALIBRATION WITH REAL CWC DISCHARGE DATA
#
# ================================================================
# CELL 13: CALIBRATION WITH REAL CWC DISCHARGE DATA
# Scales observed Bantwal discharge to AOI using linear area ratio
# (800/3657 = 0.2188), then runs full SCS-CN parameter search
# against the area-scaled 2014 peak discharge target.
#
# Calibration year  : 2014 (year-matched, within discharge record)
# CN multiplier cap : 1.50 (physical realism)
# Ia ratio          : 0.05 (pre-saturated Western Ghats monsoon)
# Area scaling      : Q_AOI = Q_Bantwal x (800/3657) = 864 m3/s
# ================================================================

import pandas as pd, re, json
from datetime import date, timedelta

# ── STEP 1: Parse discharge file ─────────────────────────────
# Format: row_number TAB FLOW_OUT_DOY_YEAR TAB discharge_m3s
# Row numbers are sequential (1-12783) across all years.
# Year is embedded in the record code. Days counted within year.

def parse_discharge_sequential(filepath):
    raw = []
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 3:
                parts = line.strip().split()
            if len(parts) < 3:
                continue
            try:
                m = re.match(r'FLOW_OUT_(\d+)_(\d+)', parts[1])
                if not m:
                    continue
                raw.append((int(parts[0]),
                             int(m.group(2)),
                             float(parts[2])))
            except:
                continue

    records, current_year, day_in_year = [], None, 0
    for seq, year, q in raw:
        if year != current_year:
            current_year, day_in_year = year, 0
        day_in_year += 1
        try:
            dt = date(year, 1, 1) + timedelta(days=day_in_year-1)
            records.append({
                'date' : dt,
                'Q_obs': q,
                'year' : dt.year,
                'month': dt.month
            })
        except:
            continue

    return pd.DataFrame(records).sort_values(
        'date').reset_index(drop=True)

obs = parse_discharge_sequential(DISCHARGE_PATH)
print(f'Parsed {len(obs):,} records: '
      f'{obs["date"].min()} to {obs["date"].max()}')

month_counts = obs['month'].value_counts().sort_index()
print(f'Records per month: {month_counts.min()}-'
      f'{month_counts.max()} (should be ~988-1085)')

# ── STEP 2: Area scaling ──────────────────────────────────────
A_full = FULL_BASIN_KM2   # 3657 km²
A_aoi  = AOI_AREA_KM2     # 800 km²
scale  = A_aoi / A_full   # 0.2188

obs_scaled             = obs.copy()
obs_scaled['Q_scaled'] = obs_scaled['Q_obs'] * scale

# 2014 monsoon statistics (scaled to AOI)
obs_2014_scaled = obs_scaled[
    (obs_scaled['year'] == 2014) &
    (obs_scaled['month'].isin([6,7,8,9]))
]
Q_2014_peak_scaled = obs_2014_scaled['Q_scaled'].max()
Q_2014_mean_scaled = obs_2014_scaled['Q_scaled'].mean()

# 35-yr long-term statistics (scaled)
monsoon_scaled = obs_scaled[
    obs_scaled['month'].isin([6,7,8,9])].copy()
monsoon_stats_scaled = monsoon_scaled.groupby('year').agg(
    Q_mean=('Q_scaled','mean'),
    Q_peak=('Q_scaled','max')
).reset_index()

OBS_MEAN_Q    = monsoon_stats_scaled['Q_mean'].mean()
OBS_PEAK_MEAN = monsoon_stats_scaled['Q_peak'].mean()
OBS_2014_PEAK = obs_2014_scaled['Q_obs'].max()  # unscaled for reference

print(f'\nArea scaling: {A_aoi:.0f}/{A_full:.0f} = {scale:.4f}')
print(f'2014 scaled peak Q  : {Q_2014_peak_scaled:.0f} m3/s')
print(f'2014 scaled mean Q  : {Q_2014_mean_scaled:.0f} m3/s')
print(f'35-yr scaled mean Q : {OBS_MEAN_Q:.0f} m3/s')

# ── STEP 3: Download and process 2014 CHIRPS ─────────────────
import gzip, shutil
from rasterio.mask import mask as rio_mask
from rasterio.warp import reproject, Resampling as WarpResampling
from shapely.geometry import box, mapping

os.makedirs(RAIN_DIR_2014, exist_ok=True)
BASE_URL = (
    'https://data.chc.ucsb.edu/products/CHIRPS-2.0/'
    'global_daily/tifs/p05/{year}/'
    'chirps-v2.0.{year}.{month:02d}.{day:02d}.tif.gz'
)

dates_2014 = []
d = date(2014, 6, 1)
while d <= date(2014, 9, 30):
    dates_2014.append(d)
    d += timedelta(days=1)

existing = len(glob.glob(f'{RAIN_DIR_2014}/*.tif.gz'))
if existing >= 122:
    print(f'2014 CHIRPS already downloaded ({existing} files)')
else:
    print(f'Downloading 2014 CHIRPS ({len(dates_2014)} files)...')
    success, failed = 0, []
    for d in dates_2014:
        url   = BASE_URL.format(
            year=d.year, month=d.month, day=d.day)
        fname = (f'chirps-v2.0.{d.year}.'
                 f'{d.month:02d}.{d.day:02d}.tif.gz')
        fpath = f'{RAIN_DIR_2014}/{fname}'
        if os.path.exists(fpath):
            success += 1
            continue
        try:
            r = requests.get(url, timeout=30, stream=True)
            if r.status_code == 200:
                with open(fpath, 'wb') as f:
                    for chunk in r.iter_content(8192):
                        f.write(chunk)
                success += 1
                if success % 30 == 0:
                    print(f'  {success}/{len(dates_2014)}...')
            else:
                failed.append(str(d))
        except:
            failed.append(str(d))
    print(f'  Done: {success} ok, {len(failed)} failed')

# Process 2014 stack
rain_gz_2014 = sorted(glob.glob(f'{RAIN_DIR_2014}/*.tif.gz'))
aoi_geom     = mapping(box(MINX, MINY, MAXX, MAXY))
stack_2014   = np.zeros(
    (len(rain_gz_2014), *target_shape), dtype=np.float32)

print(f'Processing {len(rain_gz_2014)} files...')
for i, gz_path in enumerate(rain_gz_2014):
    tif_path = gz_path.replace('.tif.gz', '.tif')
    try:
        if not os.path.exists(tif_path):
            with gzip.open(gz_path, 'rb') as fi:
                with open(tif_path, 'wb') as fo:
                    shutil.copyfileobj(fi, fo)
        with rasterio.open(tif_path) as src:
            clipped, ct = rio_mask(
                src, [aoi_geom], crop=True, nodata=-9999)
            cm = src.meta.copy()
            cm.update({'height':clipped.shape[1],
                       'width':clipped.shape[2],
                       'transform':ct,'nodata':-9999})
        tmp = tif_path.replace('.tif','_tmp.tif')
        with rasterio.open(tmp,'w',**cm) as t:
            t.write(clipped)
        with rasterio.open(tmp) as src:
            data = np.zeros(target_shape, dtype=np.float32)
            reproject(source=rasterio.band(src,1),
                      destination=data,
                      src_transform=src.transform,
                      src_crs=src.crs,
                      dst_transform=meta_utm['transform'],
                      dst_crs=DST_CRS,
                      resampling=WarpResampling.bilinear)
        data[data<0] = 0
        stack_2014[i] = data
        os.remove(tmp)
    except:
        stack_2014[i] = 0

total_2014 = stack_2014.sum(axis=0)[dem_valid].mean()
print(f'2014 seasonal total: {total_2014:.0f}mm')
np.save(f'{PROC_DIR}/rainfall_stack_2014.npy', stack_2014)

# ── STEP 4: Calibration search ────────────────────────────────
OBS_TARGET = Q_2014_peak_scaled  # 864 m3/s

def compute_sim_peak_Q(cn_grid, ia_ratio, rain_stack,
                        dem_valid, A_km2=AOI_AREA_KM2):
    """Peak daily discharge from SCS-CN over AOI."""
    daily_Q = []
    for i in range(rain_stack.shape[0]):
        r     = match_shape(rain_stack[i], target_shape)
        r     = np.where(r < 0, 0, r)
        Q_mm  = scscn_runoff(
            r, cn_grid, Ia_ratio=ia_ratio)[dem_valid].mean()
        Q_m3s = (Q_mm/1000.0) * (A_km2*1e6) / 86400.0
        daily_Q.append(Q_m3s)
    return np.max(daily_Q)

CN_multipliers = [0.85,0.90,0.95,1.00,1.05,1.10,
                  1.15,1.20,1.25,1.30,1.35,1.40,1.45,1.50]
Ia_ratios      = [0.05,0.10,0.15,0.20]

print(f'\nCalibration target: {OBS_TARGET:.0f} m3/s')
print(f'{"CN_mult":>8} {"Ia":>6} {"Sim_Qp":>10} '
      f'{"Obs_Qp":>8} {"PBIAS%":>8} {"NSE":>8}')
print('-'*55)

best_nse, best_params = -999, (1.50, 0.05)

for cn_m in CN_multipliers:
    for ia in Ia_ratios:
        cn_adj  = np.clip(cn * cn_m, 1, 99)
        sim_Qp  = compute_sim_peak_Q(
            cn_adj, ia, stack_2014, dem_valid)
        pbias   = (sim_Qp - OBS_TARGET) / OBS_TARGET * 100
        nse     = 1 - ((sim_Qp-OBS_TARGET)**2/OBS_TARGET**2)
        print(f'{cn_m:>8.2f} {ia:>6.2f} {sim_Qp:>10,.0f} '
              f'{OBS_TARGET:>8,.0f} {pbias:>8.1f}% {nse:>8.3f}')
        if nse > best_nse:
            best_nse    = nse
            best_params = (cn_m, ia)
            best_Qp     = sim_Qp

# ── STEP 5: Apply best parameters ────────────────────────────
best_cn_m, best_ia = best_params
cn_calibrated    = np.clip(cn * best_cn_m, 1, 99)
Q_calibrated     = scscn_runoff(
    rain_event, cn_calibrated, Ia_ratio=best_ia)
flood_calibrated = estimate_flood_depth(
    Q_calibrated, hand, slope, facc, river)
save_flood_raster(flood_calibrated,
                  'flood_depth_calibrated.tif', meta_utm)
np.save(f'{PROC_DIR}/flood_depth_calibrated.npy', flood_calibrated)

pbias_final = (best_Qp - OBS_TARGET) / OBS_TARGET * 100

# Save summary
calib_summary = {
    'method'              : 'SCS-CN peak discharge',
    'calib_year'          : 2014,
    'area_scaling'        : 'linear (A_aoi/A_full)',
    'A_aoi_km2'           : float(A_aoi),
    'A_full_km2'          : float(A_full),
    'scale_factor'        : float(scale),
    'cn_multiplier'       : float(best_cn_m),
    'ia_ratio'            : float(best_ia),
    'Q_sim_m3s'           : float(best_Qp),
    'Q_obs_full_m3s'      : float(OBS_2014_PEAK),
    'Q_obs_scaled_m3s'    : float(OBS_TARGET),
    'PBIAS_pct'           : float(pbias_final),
    'NSE'                 : float(best_nse),
}
with open(f'{PROC_DIR}/calibration_summary.json','w') as f:
    json.dump(calib_summary, f, indent=2)

print(f'\n{"="*55}')
print(f'CALIBRATION RESULT')
print(f'{"="*55}')
print(f'  Calibration year        : 2014 (year-matched)')
print(f'  Observed Q (Bantwal)    : {OBS_2014_PEAK:,.0f} m3/s')
print(f'  Observed Q (AOI-scaled) : {OBS_TARGET:.0f} m3/s')
print(f'  Simulated peak Q        : {best_Qp:.0f} m3/s')
print(f'  Best CN multiplier      : {best_cn_m:.2f}')
print(f'  Best Ia ratio           : {best_ia:.2f}')
print(f'  PBIAS                   : {pbias_final:.1f}%')
print(f'  NSE                     : {best_nse:.3f}')
print(f'\nRun Cell 14 next')

In [ ]:
# ================================================================
# CELL 14: MULTI-SCENARIO FLOOD SIMULATION + THRESHOLD MAP
# Runs calibrated model at 7 rainfall multipliers.
# Produces rainfall threshold map: minimum rainfall per pixel
# needed to cause flood depth > 0.1m.
# ================================================================

print(f'Calibrated params: CN x{best_cn_m:.2f}, Ia={best_ia:.2f}')
print('Running scenarios...')
scenarios  = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
flood_maps = {}

for mult in scenarios:
    rain_s  = rain_event * mult
    Q_s     = scscn_runoff(rain_s, cn_calibrated, Ia_ratio=best_ia)
    flood_s = estimate_flood_depth(Q_s, hand, slope, facc, river)
    flood_maps[mult] = flood_s
    fname   = f'flood_depth_{str(mult).replace(".","p")}x.tif'
    save_flood_raster(flood_s, fname, meta_utm)
    pct = (flood_s>0.1).sum()/dem_valid.sum()*100
    print(f'  {mult:.2f}x  max:{np.nanmax(flood_s):.3f}m  flooded:{pct:.1f}%')

# Rainfall threshold map
print('\nBuilding threshold map...')
threshold_map = np.full(target_shape, np.nan)
for mult in sorted(scenarios):
    just_flooded = (
        np.isnan(threshold_map) &
        (flood_maps[mult] >= 0.1) & dem_valid
    )
    threshold_map[just_flooded] = mult * rain_event[just_flooded]

never_flooded = np.isnan(threshold_map) & dem_valid
threshold_map[never_flooded] = rain_event[never_flooded] * 2.0
threshold_map[~dem_valid]    = np.nan

out_meta = meta_utm.copy()
out_meta.update({'dtype':'float32','nodata':-9999,'count':1})
thresh_arr = np.where(np.isnan(threshold_map),-9999,threshold_map).astype('float32')
with rasterio.open(f'{OUTPUTS_DIR}/rainfall_threshold_map.tif','w',**out_meta) as dst:
    dst.write(thresh_arr,1)

# GeoJSON
with rasterio.open(f'{OUTPUTS_DIR}/rainfall_threshold_map.tif') as src:
    transform = src.transform
rows_idx, cols_idx = np.where(dem_valid & ~np.isnan(threshold_map))
features = []
for r,c in zip(rows_idx[::10][:5000], cols_idx[::10][:5000]):
    lon,lat = rasterio.transform.xy(transform, r, c)
    features.append({'type':'Feature',
                     'geometry':{'type':'Point','coordinates':[lon,lat]},
                     'properties':{'threshold_mm':round(float(threshold_map[r,c]),1),
                                   'hand_m':round(float(hand[r,c]),1),
                                   'cn':round(float(cn_calibrated[r,c]),1)}})
with open(f'{OUTPUTS_DIR}/rainfall_threshold_map.geojson','w') as f:
    json.dump({'type':'FeatureCollection','features':features},f)

np.save(f'{PROC_DIR}/threshold_map.npy', threshold_map)
valid_t = threshold_map[dem_valid & ~np.isnan(threshold_map)]
print(f'Threshold map: {valid_t.min():.0f}-{valid_t.max():.0f}mm')
print(f'GeoJSON: {len(features):,} points')

In [ ]:
# ================================================================
# CELL 15: VISUALISATION — FLOOD MODEL OUTPUTS
# 6-panel figure showing all key outputs.
# ================================================================

fig, axes = plt.subplots(2,3,figsize=(16,10))
fig.patch.set_facecolor('#0d1117')
fig.suptitle(
    f'SCS-CN Flood Model v2.0 — Upper Nethravathi Basin\n'
    f'Calibration: 2014 year-matched | CN x{best_cn_m:.2f}, '
    f'Ia={best_ia:.2f} | PBIAS={pbias_final:.0f}%',
    color='white', fontsize=10, fontweight='bold'
)

panels = [
    (flood_calibrated, 'Blues',    f'Flood Depth — Aug 2018 (m)'),
    (Q_calibrated,     'YlOrRd',   'SCS-CN Runoff Depth (mm)'),
    (threshold_map,    'RdYlGn_r', 'Rainfall Flood Threshold (mm)'),
    (hand,             'terrain',  'HAND — m above channel'),
    (flood_maps[2.0],  'Blues',    '2× Rainfall Scenario (m)'),
    (flood_maps[0.5],  'Blues',    '0.5× Rainfall Scenario (m)'),
]

for ax,(data,cmap,title) in zip(axes.flat, panels):
    ax.set_facecolor('#0d1117')
    d2   = np.where(np.isnan(data),np.nan,data)
    vmax = np.nanpercentile(d2[~np.isnan(d2)], 98)
    im   = ax.imshow(d2, cmap=cmap, origin='upper', vmin=0, vmax=vmax)
    cb   = plt.colorbar(im, ax=ax, shrink=0.75)
    cb.ax.yaxis.set_tick_params(color='white', labelsize=7)
    plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')
    ax.set_title(title, color='white', fontsize=9)
    ax.tick_params(colors='#666', labelsize=7)

plt.tight_layout()
out = f'{OUTPUTS_DIR}/plots/flood_model_v2.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'Saved: {out}')

In [ ]:
# ================================================================
# CELL 16: GNN — GRAPH CONSTRUCTION
# Converts raster data to a graph for the GCN.
# Nodes = pixels (sampled every 6th), Edges = k=8 neighbours.
#
# Node features (6):
#   elevation, slope, HAND, flow_accumulation (log),
#   distance to river, Aug 8-18 rainfall
#
# Target: calibrated flood depth (m)
# ================================================================

!pip install torch torch-geometric --quiet
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial import cKDTree

dist_river, _ = load_raster(f'{PROC_DIR}/dist_to_river.tif')
dist_river    = match_shape(dist_river, target_shape)
dist_river    = np.where(np.isnan(dist_river),
                          np.nanmean(dist_river), dist_river)

STEP = 6
rows, cols   = np.where(dem_valid)
sample_mask  = (rows%STEP==0) & (cols%STEP==0)
rows_s, cols_s = rows[sample_mask], cols[sample_mask]
n_nodes      = len(rows_s)
print(f'Graph nodes: {n_nodes:,} (every {STEP}th pixel)')

X = np.stack([
    dem_utm[rows_s, cols_s],
    slope[rows_s, cols_s],
    hand[rows_s, cols_s],
    facc[rows_s, cols_s],
    dist_river[rows_s, cols_s],
    rain_event[rows_s, cols_s]
], axis=1)
X = MinMaxScaler().fit_transform(np.nan_to_num(X, 0)).astype(np.float32)

y = np.nan_to_num(flood_calibrated[rows_s, cols_s], 0).astype(np.float32)

coords = np.stack([rows_s, cols_s], axis=1).astype(np.float32)
_, idx = cKDTree(coords).query(coords, k=9)
idx    = idx[:, 1:]

edge_index = torch.tensor(
    np.stack([np.repeat(np.arange(n_nodes),8), idx.flatten()]),
    dtype=torch.long
)

graph = Data(
    x          = torch.tensor(X, dtype=torch.float),
    edge_index = edge_index,
    y          = torch.tensor(y, dtype=torch.float).unsqueeze(1)
)

print(f'Nodes: {graph.num_nodes:,} | Edges: {graph.num_edges:,} | '
      f'Features: {graph.num_node_features}')
print(f'Target mean: {y.mean():.4f}m | max: {y.max():.4f}m')

In [ ]:
# ================================================================
# CELL 17: GNN — MODEL DEFINITION
# 2-layer Graph Convolutional Network (GCN).
# Architecture: Input(6) → GCNConv(64) → GCNConv(32) → Linear(1)
# LeakyReLU activations, gradient clipping during training.
# ================================================================

import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class FloodGCN(nn.Module):
    def __init__(self, in_channels, h1=64, h2=32):
        super().__init__()
        self.conv1 = GCNConv(in_channels, h1)
        self.conv2 = GCNConv(h1, h2)
        self.fc    = nn.Linear(h2, 1)

    def forward(self, x, edge_index):
        x = F.leaky_relu(self.conv1(x, edge_index), 0.2)
        x = F.leaky_relu(self.conv2(x, edge_index), 0.2)
        return self.fc(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = FloodGCN(in_channels=graph.num_node_features).to(device)

print(f'GCN model: {sum(p.numel() for p in model.parameters()):,} parameters')
print(f'Device   : {device}')
print(f'Architecture: Input(6) -> GCNConv(64) -> GCNConv(32) -> Linear(1)')

In [ ]:
# ================================================================
# CELL 18: GNN — TRAINING
# 80/20 train/test split. 200 epochs.
# Adam lr=0.001, MSELoss, gradient clipping max_norm=1.0.
# Best model saved by test loss.
# ================================================================

from sklearn.model_selection import train_test_split

idx_train, idx_test = train_test_split(
    np.arange(graph.num_nodes), test_size=0.2, random_state=42)

train_mask = torch.zeros(graph.num_nodes, dtype=torch.bool)
test_mask  = torch.zeros(graph.num_nodes, dtype=torch.bool)
train_mask[idx_train] = True
test_mask[idx_test]   = True

graph.train_mask, graph.test_mask = train_mask, test_mask
graph_d   = graph.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.MSELoss()
EPOCHS    = 200
train_losses, test_losses = [], []
best_test_loss = float('inf')

print(f'Training: {EPOCHS} epochs | '
      f'Train={train_mask.sum():,} Test={test_mask.sum():,}')

for epoch in range(1, EPOCHS+1):
    model.train()
    optimizer.zero_grad()
    out  = model(graph_d.x, graph_d.edge_index)
    tp   = out[graph_d.train_mask]
    tt   = graph_d.y[graph_d.train_mask]
    vm   = ~torch.isnan(tp) & ~torch.isnan(tt)
    if vm.sum() == 0: continue
    loss = criterion(tp[vm], tt[vm])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    train_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        ep  = out[graph_d.test_mask]
        et  = graph_d.y[graph_d.test_mask]
        evm = ~torch.isnan(ep) & ~torch.isnan(et)
        if evm.sum()>0:
            tl = criterion(ep[evm], et[evm]).item()
            test_losses.append(tl)
            if tl < best_test_loss:
                best_test_loss = tl
                torch.save(model.state_dict(),
                           f'{PROJECT_ROOT}/models/best_gcn.pt')

    if epoch % 25 == 0:
        print(f'  Epoch {epoch:>3}  '
              f'Train={loss.item():.6f}  Test={tl:.6f}')

print(f'\nTraining complete. Best test loss: {best_test_loss:.6f}')

In [ ]:
# ================================================================
# CELL 19: GNN — EVALUATION
# Loads best model. Reports RMSE, MAE, R².
# Plots: predicted vs observed, loss curve, spatial prediction map.
# ================================================================

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

model.load_state_dict(torch.load(f'{PROJECT_ROOT}/models/best_gcn.pt'))
model.eval()

with torch.no_grad():
    out   = model(graph_d.x, graph_d.edge_index)
    preds = out[graph_d.test_mask].cpu().numpy().flatten()
    truth = graph_d.y[graph_d.test_mask].cpu().numpy().flatten()

valid = ~np.isnan(preds) & ~np.isnan(truth)
preds, truth = preds[valid], truth[valid]

rmse = np.sqrt(mean_squared_error(truth, preds))
mae  = mean_absolute_error(truth, preds)
r2   = r2_score(truth, preds)

print('GNN EVALUATION')
print('='*40)
print(f'  RMSE : {rmse:.4f} m')
print(f'  MAE  : {mae:.4f} m')
print(f'  R²   : {r2:.4f}')
print(f'\n  R²>0.4 = acceptable | R²>0.6 = good | R²>0.8 = excellent')

# Plot
fig, axes = plt.subplots(1,3,figsize=(15,5))
fig.patch.set_facecolor('#0d1117')
fig.suptitle(f'GNN Flood Depth Prediction — R²={r2:.3f}, RMSE={rmse:.4f}m',
             color='white', fontsize=11, fontweight='bold')

ax1 = axes[0]; ax1.set_facecolor('#1a1d27')
ax1.scatter(truth, preds, alpha=0.3, s=1, color='#4fa3e0')
lim = max(truth.max(), preds.max())
ax1.plot([0,lim],[0,lim],'r--',lw=1,label='1:1')
ax1.set_xlabel('Observed (m)',color='white'); ax1.set_ylabel('Predicted (m)',color='white')
ax1.set_title(f'Predicted vs Observed\nR²={r2:.3f}',color='white',fontsize=9)
ax1.tick_params(colors='white'); ax1.legend(labelcolor='white',facecolor='#2a2d3a')

ax2 = axes[1]; ax2.set_facecolor('#1a1d27')
ax2.plot(train_losses,color='#4fa3e0',lw=1,label='Train')
ax2.plot(test_losses, color='#e74c3c',lw=1,label='Test')
ax2.set_xlabel('Epoch',color='white'); ax2.set_ylabel('MSE Loss',color='white')
ax2.set_title('Training Curve',color='white',fontsize=9)
ax2.tick_params(colors='white'); ax2.legend(labelcolor='white',facecolor='#2a2d3a')

with torch.no_grad():
    all_out = model(graph_d.x, graph_d.edge_index).cpu().numpy().flatten()
pred_map = np.full(target_shape, np.nan)
pred_map[rows_s, cols_s] = all_out

ax3 = axes[2]; ax3.set_facecolor('#0d1117')
vmax = np.nanpercentile(pred_map[~np.isnan(pred_map)], 98)
im   = ax3.imshow(pred_map,cmap='Blues',origin='upper',vmin=0,vmax=vmax)
cb   = plt.colorbar(im,ax=ax3,shrink=0.8)
cb.set_label('Depth (m)',color='white')
cb.ax.yaxis.set_tick_params(color='white')
plt.setp(cb.ax.yaxis.get_ticklabels(),color='white')
ax3.set_title('GNN Predicted Flood Depth',color='white',fontsize=9)
ax3.tick_params(colors='#666')

plt.tight_layout()
out_path = f'{OUTPUTS_DIR}/plots/gnn_evaluation.png'
plt.savefig(out_path,dpi=150,bbox_inches='tight',facecolor='#0d1117')
plt.show()

# Save GNN prediction raster
pm = meta_utm.copy()
pm.update({'dtype':'float32','nodata':-9999,'count':1})
pa = np.where(np.isnan(pred_map),-9999,pred_map).astype('float32')
with rasterio.open(f'{OUTPUTS_DIR}/gnn_flood_prediction.tif','w',**pm) as dst:
    dst.write(pa,1)
print(f'\nSaved: {out_path}')
print(f'Saved: {OUTPUTS_DIR}/gnn_flood_prediction.tif')

In [ ]:
# ── Generate August 2018 discharge hydrograph ─────────────────
from datetime import date, timedelta

A_km2      = AOI_AREA_KM2   # 800 km²
dates_aug  = [date(2018,8,1) + timedelta(days=i) for i in range(31)]
date_labels = [d.strftime("%b %d") for d in dates_aug]

# Daily runoff for August 2018 (days 61-91 from Jun 1)
daily_Q_m3s = []
for i in range(61, 92):
    r     = match_shape(rainfall_stack[i], target_shape)
    r     = np.where(r < 0, 0, r)
    Q_mm  = scscn_runoff(r, cn_calibrated, Ia_ratio=best_ia)[dem_valid].mean()
    Q_m3s = (Q_mm / 1000.0) * (A_km2 * 1e6) / 86400.0
    daily_Q_m3s.append(Q_m3s)

daily_rain = [rainfall_stack[61+i][dem_valid].mean() for i in range(31)]
Q_obs_scaled_linear = OBS_2014_PEAK * (AOI_AREA_KM2 / FULL_BASIN_KM2)
# = 3949 × (800/3657) = 864 m³/s
print(f'Scaled observed peak Q: {Q_obs_scaled_linear:.0f} m3/s')
# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
fig.patch.set_facecolor('#0d1117')

ax1.set_facecolor('#1a1d27')
ax1.plot(range(31), daily_Q_m3s, color='#4fa3e0', lw=2.5)
ax1.fill_between(range(31), 0, daily_Q_m3s, alpha=0.3, color='#4fa3e0')
ax1.axhline(Q_obs_scaled_linear, color='#e74c3c', lw=1.5,
            linestyle='--', label=f'Scaled obs peak: {Q_obs_scaled_linear:.0f} m\u00b3/s')
ax1.axvline(7, color='#f39c12', lw=1, linestyle=':',
            label='Aug 8 (event start)')
ax1.axvline(17, color='#f39c12', lw=1, linestyle=':',
            label='Aug 18 (event end)')
ax1.set_ylabel('Discharge (m\u00b3/s)', color='white')
ax1.set_title('Simulated Daily Discharge — August 2018\nUpper Nethravathi AOI (~800 km\u00b2)',
              color='white', fontsize=10)
ax1.set_xticks(range(0, 31, 5))
ax1.set_xticklabels([date_labels[i] for i in range(0, 31, 5)],
                     color='white', fontsize=8)
ax1.tick_params(colors='white')
ax1.legend(facecolor='#2a2d3a', labelcolor='white', fontsize=8)
for sp in ax1.spines.values(): sp.set_edgecolor('#333')

ax2.set_facecolor('#1a1d27')
ax2.bar(range(31), daily_rain, color='#4fa3e0', alpha=0.7)
ax2.set_ylabel('Rainfall (mm/day)', color='white')
ax2.set_title('Basin-mean Daily Rainfall (CHIRPS)', color='white', fontsize=10)
ax2.set_xticks(range(0, 31, 5))
ax2.set_xticklabels([date_labels[i] for i in range(0, 31, 5)],
                     color='white', fontsize=8)
ax2.tick_params(colors='white')
for sp in ax2.spines.values(): sp.set_edgecolor('#333')

plt.tight_layout()
out = f'{OUTPUTS_DIR}/plots/discharge_hydrograph.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(f'Peak simulated Q : {max(daily_Q_m3s):.0f} m\u00b3/s')
print(f'Peak day         : {date_labels[daily_Q_m3s.index(max(daily_Q_m3s))]}')
print(f'Saved: {out}')

In [ ]:
# ================================================================
# CELL 20 FIXED: BACKUP TO GOOGLE DRIVE
# Copies files one by one with error handling.
# Skips large numpy stacks (regenerated from CHIRPS if needed).
# ================================================================

from google.colab import drive
import shutil, os, glob

# Remount Drive cleanly
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

BACKUP = '/content/drive/MyDrive/Nethravathi_Flood_Backup_v2'
os.makedirs(BACKUP, exist_ok=True)

# Files to skip — too large or regenerable from CHIRPS
SKIP_FILES = {
    'rainfall_stack.npy',       # 1.5GB — regenerate from CHIRPS
    'rainfall_stack_2014.npy',  # 1.5GB — regenerate from CHIRPS
    'total_rainfall.npy',       # regenerable
}

def safe_copy_folder(src_folder, dst_folder, skip=set()):
    os.makedirs(dst_folder, exist_ok=True)
    ok, skipped, failed = [], [], []

    for item in os.listdir(src_folder):
        if item in skip:
            skipped.append(item)
            continue
        src  = os.path.join(src_folder, item)
        dst  = os.path.join(dst_folder, item)

        if os.path.isdir(src):
            safe_copy_folder(src, dst, skip)
            continue

        try:
            shutil.copy2(src, dst)
            size = os.path.getsize(dst) / 1e6
            ok.append(f'{item} ({size:.1f}MB)')
        except Exception as e:
            failed.append(f'{item}: {str(e)[:60]}')

    return ok, skipped, failed

print('Backing up to Google Drive...\n')

# Processed data (skip large stacks)
print('[1/3] Processed data...')
ok, sk, fail = safe_copy_folder(
    PROC_DIR,
    f'{BACKUP}/processed',
    skip=SKIP_FILES
)
print(f'  Copied : {len(ok)} files')
print(f'  Skipped: {sk}')
if fail: print(f'  Failed : {fail}')

# Outputs
print('\n[2/3] Outputs...')
ok, sk, fail = safe_copy_folder(
    OUTPUTS_DIR,
    f'{BACKUP}/outputs',
    skip=SKIP_FILES
)
print(f'  Copied : {len(ok)} files')
if fail: print(f'  Failed : {fail}')

# Models
print('\n[3/3] Models...')
ok, sk, fail = safe_copy_folder(
    f'{PROJECT_ROOT}/models',
    f'{BACKUP}/models',
    skip=SKIP_FILES
)
print(f'  Copied : {len(ok)} files')
if fail: print(f'  Failed : {fail}')

# List what was saved
print('\nBacked up to:', BACKUP)
print('\nFiles in backup:')
for root, dirs, files in os.walk(BACKUP):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)/1e6
        rel  = path.replace(BACKUP, '')
        print(f'  {rel:<60} {size:.1f}MB')

print('\nNote: rainfall_stack.npy files skipped (too large)')
print('These are regenerated by Cell 11 from CHIRPS downloads.')